# 01 · Sessions setup

**What this notebook does.** It decides **which recordings and which trials** every later notebook uses. It reads the raw
and processed folders (read-only), checks every session, and builds the tables that notebooks 02–07 read:
the final session list, the filtered trial table and the brain region of every recorded channel.

**Run order in the project:** **01** → 02 preprocessing → 03 behaviour → 04 QC → 05 event check → 06 analysis → 07 results.
01 must have been run once (with saving on) before any other notebook can work, because the others read its tables.

**Steps in this notebook**

| Step | What it does | Main output table (in `TABLES`) |
|---|---|---|
| 1 | Data inventory: what one session contains, how many sessions have data | – |
| 2 | Session list: every item in `0_raw`, which ones are usable | `session_inventory.csv`, `usable_sessions.csv` |
| 3 | Is every 1000 Hz recording complete? | – |
| 4 | Session flags from the task settings (short/long design, WT manipulation) | `session_flags.csv` |
| 5 | Sync details (behaviour clock ↔ neural clock) | `sync_summary.csv` |
| 6 | **Final session list** (60 sessions) | **`final_sessions.csv`** |
| 7 | **Trial filter**: labels, neural-clock times, exclusion rules R1–R5 | **`trials_filtered.csv`**, `trial_funnel.csv` |
| 8 | Electrode maps and depth coverage | `electrode_maps.csv`, `depth_coverage.csv` |
| 9 | Task settings and tone delays per session | `task_settings_per_session.csv`, `tone_delay_longprob_check.csv` |
| 10 | **Probe location**: brain region of every recorded channel, checks and figures | **`channel_regions_all.csv`**, `channel_regions_inventory.csv` |
| 11 | Histology previews and the recoloured atlas figure | figures only |

Tables in **bold** are read by later notebooks.

**How to use it**
1. Run the **settings cell** first (after every kernel restart). Every other cell needs only the settings cell, so any
   cell can be run on its own, in any order.
2. Each step reads its input table from `TABLES` (written by an earlier step). If you change something in an early step
   (e.g. a trial rule), run that step with `SAVE = True`, then the steps after it.
3. **Nothing is written to disk unless `SAVE = True`** in the settings. With `SAVE = False` you can run everything
   and only look at the results.

**What to change, and where:** only in the settings cell (paths, example session, task constants, trial rules).
The functions behind the cells are in `fx_sessions.py`; shared colours and the save functions are in `fx_common.py`.

## Settings

Run this cell first. It defines everything the other cells need.

**Paths**
- `CODE_DIR`: the folder with the notebooks and the `fx_*.py` files (this repository's `main` folder).
- `DATA_ROOT`: the project data folder on the server. `RAW` (`0_raw`, raw recordings) and `PROC` (`2_processed`,
  down-sampled recordings + behaviour in `share`) are inside it and are **only read**.
- `AMIR`: your own data folder (`amir_data`). Everything this project writes goes inside it.
- `TABLES`: where the session and trial tables are read from and written to. Every later notebook reads them here.
- `FIG_DIR`: where the figures of this notebook go when saving is on.
- `PROBE_DIR`: where the probe-position table and figure go (step 10d).
- `HIST_SRC`, `HIST_DIR`: histology slides (`.czi`) and where their previews go. `ATLAS_SRC`, `ATLAS_OUT`: the atlas image
  to recolour and its output name (step 11b).

**Saving**
- `SAVE = False`: nothing is written. `SAVE = True`: every figure goes to `FIG_DIR` (PNG) and every table to `TABLES`.
- **To save one cell somewhere else**, add `folder=` (and `force=True` to save even when `SAVE` is False) in that cell's
  save line, e.g. `fxc.save_table(final, "final_sessions", folder=r"D:\my_folder", force=True)`.

**Task constants** (change only if the task changes)
- `SHORT_WT`, `LONG_WT`: waiting time of short and long trials (s). Every other value = a WT-manipulation session.
- `TONE_DELAY`: measured delay from press to tone for each waiting time (s). It is 0.35 / 1.05 s, not the nominal 0.3 / 1.0.
- `EARLY_MAX_HOLD`: rule R3 — an "early" trial held longer than this (>) is excluded (it is really a late/correct trial).
- `RESP_WIN`: response window after the tone (s). A correct trial must be released within it (rule R5).

**Other**
- `EXAMPLE`: the session shown in step 1. `BIN_UM`, `MIN_SESS`: depth-bin size and the minimum number of sessions per bin (step 8).
- `X_OFFSET_UM`: the channel-region files store x = probe x + 27 µm; used to match channels by position (step 10).
- `read(name)`: reads `TABLES/name.csv` with the rat column as text.

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

# ── paths: change these to your computer ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server (read-only)
RAW       = os.path.join(DATA_ROOT, "0_raw")                                       # raw recordings (ap.bin, ap.meta)
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings + share/trials.csv
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder (all outputs)
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
FIG_DIR   = os.path.join(AMIR, "figures", "01_sessions_setup")                     # figures of this notebook (if SAVE)
PROBE_DIR = os.path.join(AMIR, "Probe")                                            # probe-position table + figure (step 10d)
HIST_SRC  = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\Histology\134 brain\4Ahmed"   # .czi slides (step 11)
HIST_DIR  = os.path.join(AMIR, "histology")                                        # slide previews (if SAVE)
ATLAS_SRC = os.path.join(os.path.expanduser("~"), "Desktop", "histology.png")      # atlas image to recolour (step 11b)
ATLAS_OUT = "histology_mPFC_recolored"                                             # its output name (saved in HIST_DIR)

# ── saving: False = nothing is written ──
SAVE = False                                                                       # True = write figures (FIG_DIR) and tables (TABLES)

# ── task constants: change only if the task changes ──
SHORT_WT, LONG_WT = 0.3, 1.0                                                       # waiting time of short / long trials (s)
TONE_DELAY        = {0.3: 0.35, 1.0: 1.05}                                         # measured press → tone delay (s) per waiting time
EARLY_MAX_HOLD    = 1.05                                                           # R3: "early" held longer than this (>) is excluded (s)
RESP_WIN          = 0.5                                                            # R5: correct = release within this after the tone (s)

# ── other ──
EXAMPLE     = "20260529_r0000136_wt_1318"                                          # session shown in step 1 (any session)
BIN_UM      = 100                                                                  # depth bin for the coverage map (µm)
MIN_SESS    = 5                                                                    # min sessions per bin to count a bin as testable
X_OFFSET_UM = 27                                                                   # channel-region files: x = probe x + 27 µm

# ── code ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, fx_sessions as fs
importlib.reload(fxc); importlib.reload(fs)
read = lambda name: pd.read_csv(os.path.join(TABLES, name + ".csv"), dtype={"rat": str})   # read one table from TABLES
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

fxc.check_paths(RAW=RAW, PROC=PROC, AMIR=AMIR)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)
print("tables found in TABLES:", sorted(f for f in os.listdir(TABLES) if f.endswith(".csv")) if os.path.isdir(TABLES) else "TABLES does not exist yet")

## Step 1 · Data inventory

**What it does.** Shows the two things every analysis uses, for one example session (`EXAMPLE`), and counts how many sessions have them:
1. `2_processed\<session>\amir_data\down_sampled\1000_Hz`: the neural data, down-sampled to 1000 Hz, no reference.
   The recording is opened with spikeinterface, which reads only the metadata (number of channels, duration), not the traces.
2. `2_processed\<session>\share`: the behaviour (`trials.csv`) and the sync information (`session_info.json`).
   The min/max of the neural-clock event columns are shown so you can see they fall inside the recording.

Then it counts the folders in `2_processed`, how many have a 1000 Hz folder, a `DONE.txt` (the down-sampling finished)
and a `share/trials.csv`.

**Expected:** 383 channels (1 of 384 removed), 32.1 min for the example; one session without `DONE.txt`
(`20260522_r0000136_wt_1750`, the old pilot, which is complete; see step 3).

In [ ]:
counts = fs.data_inventory(PROC, EXAMPLE)

## Step 2 · Session list

**What it does.** Goes through every item in `0_raw` and records:
- the number and size of `ap.bin` files (raw recording);
- whether `share\trials.csv` exists and whether it is **synced** (it has the column `press_np_s` = press time on the neural clock);
- whether the 1000 Hz recording is finished (`DONE.txt`);
- the electrode map (`~imroTbl` in `ap.meta`), turned into a number `map_id` (same number = same channels recorded);
- from `session_info.json`: `sync_ok`, the sync coverage and the waiting times of the task.

**Usable session** = has an `ap.bin` **and** a synced `trials.csv`. Each non-usable item gets a reason (no ap.bin / no trials.csv / not synced).

**Expected:** 148 items in `0_raw`, **63 usable** (rat 134: 14, 135: 29, 136: 20); `sync_ok` False in 9 usable sessions.

**Saves:** `session_inventory.csv` (all items) and `usable_sessions.csv` (usable sessions, with `map_id`).

In [ ]:
sessions, usable = fs.session_inventory(RAW, PROC)
fxc.save_table(sessions, "session_inventory")
fxc.save_table(usable, "usable_sessions")

## Step 3 · Is every 1000 Hz recording complete?

**What it does.** `DONE.txt` only says the save finished; this step checks the data itself, for every usable session:
1. **size check:** the file `traces_cached_seg0.raw` has exactly channels × samples × bytes per sample (from the metadata);
2. **end check:** the last 5 s (5000 samples) are not flat (standard deviation > 0) in any channel, so the end of the file was written.

A session is **complete** if both are true.

**Expected:** 63 of 63 complete. The old pilot has no `DONE.txt` but passes both checks, so it is kept.

**Input:** `usable_sessions.csv`. **Saves:** nothing.

In [ ]:
size_chk = fs.check_1000hz(read("usable_sessions"), PROC)

## Step 4 · Session flags from the task settings

**What it does.** Reads the `wt` column (waiting time of each trial, s) of every usable session's `trials.csv`, rounded to 0.01 s, and labels each session:
- `two_wt_session`: has both `SHORT_WT` (0.3) and `LONG_WT` (1.0) trials;
- `single_wt_session`: has only 1.0. The rat always waits 1.0 s, so short and long can't be compared in it;
- `wt_manip_session`: has any other `wt` value (WT-manipulation days).

Then two columns say what each session is used for:
- `include` = not a WT-manipulation session → used in **all** analyses;
- `short_vs_long` = included **and** has both waiting times → used in every short-vs-long comparison.

The trial type comes from `wt`, not from the `long` column of `trials.csv`.

**Expected:** 3 WT-manipulation sessions (rat 134: 23, 24 and 25 June), 7 single-wt sessions; 60 included, 53 for short vs long.

**Input:** `usable_sessions.csv`. **Saves:** `session_flags.csv`.

In [ ]:
flags = fs.session_flags(read("usable_sessions"), PROC, SHORT_WT, LONG_WT)
fxc.save_table(flags, "session_flags")

## Step 5 · Sync details

**What it does.** Reads the `sync` section of every usable session's `session_info.json`: how well the behaviour clock
was matched to the neural clock (matched pulses, coverage of the session, lag and its spread, trials outside the synced span).
It lists the sessions with `sync_ok` False (lowest coverage first) and compares the two groups (min / median / max).

**Why it matters:** in the 9 sessions with `sync_ok` False, trials outside the synced span have no reliable neural time.
They are not removed here; they are removed trial by trial in step 7 (rule R1: no press time on the neural clock).

**Input:** `usable_sessions.csv`. **Saves:** `sync_summary.csv`.

In [ ]:
sync_tab = fs.sync_details(read("usable_sessions"), PROC)
fxc.save_table(sync_tab, "sync_summary")

## Step 6 · Final session list

**What it does.** Joins the usable sessions with their flags and keeps every session with `include == True`,
i.e. all usable sessions except the WT-manipulation sessions.

- Kept: the single-wt sessions (they have `short_vs_long = False`, so they are left out of short-vs-long comparisons only).
- Kept: the old pilot (complete, step 3) and the 9 `sync_ok` False sessions (their unsynced trials are removed in step 7).

**Expected:** 63 usable − 3 excluded = **60 final sessions** (rat 134: 11, 135: 29, 136: 20); 53 of them for short vs long.

**Input:** `usable_sessions.csv`, `session_flags.csv`. **Saves:** **`final_sessions.csv`**, read by every later notebook.

In [ ]:
final = fs.final_sessions(read("usable_sessions"), read("session_flags"))
fxc.save_table(final, "final_sessions")

## Step 7 · Trial filter

**What it does.** Collects all trials of the 60 final sessions and adds:

**Labels**
- `type` from `wt`: 0.3 → short, 1.0 → long, else other_wt.
- `label` (outcome), in this order of priority: touch > correct > early > late; anything else = other.
- `prev_outcome`: the label of the previous trial in the same session. `latency_to_press_s`: time from the end of the previous trial to this press.

**Times on the neural clock**
- press = `press_ttl_np_s` (TTL pulse) if present, else `press_np_s` (synced behaviour time). `press_from_ttl` says which.
- tone and release = neural press + their delay after the press in the behaviour file (same clock, so the delay is exact).
  `tone` = 0 in `trials.csv` means no tone was played → NaN.
- `rt_s` = release − tone (reaction time; only trials with a tone).

**Exclusion rules** (a trial is removed if **any** rule applies)

| Rule | Removes | Why |
|---|---|---|
| R1 | no press time on the neural clock | can't be aligned to the LFP (mostly unsynced parts of `sync_ok` False sessions) |
| R2 | touch trials | not a lever response |
| R3 | "early" trials held longer than `EARLY_MAX_HOLD` (> 1.05 s) | held past the long tone, so not really early |
| R4 | label "other" | no defined outcome |
| R5 | timing mismatch: hold between `wt` and the tone delay, or correct with RT > `RESP_WIN` (0.5 s) | the label and the timing disagree |

All `long_prob` blocks are kept. The table keeps **all** trials, with one column per rule and a `keep` column, so later
notebooks use `keep == True`. The funnel shows how many trials are left after each rule (applied one after the other)
and how many trials are removed by only that rule.

**Expected:** 9877 trials → **8615 kept** (rat 134: 1504, 135: 4659, 136: 2452); press from TTL in 7851 kept trials.

**Input:** `final_sessions.csv`. **Saves:** **`trials_filtered.csv`** (read by every later notebook) and `trial_funnel.csv`.

In [ ]:
trials, funnel = fs.trial_filter(read("final_sessions"), PROC, TONE_DELAY, EARLY_MAX_HOLD, RESP_WIN)
fxc.save_table(trials, "trials_filtered")
fxc.save_table(funnel, "trial_funnel")

## Step 8 · Electrode maps and depth coverage

**What it does.**
- **Electrode maps:** for each `map_id` used by the final sessions, reads the probe layout of one example session
  (`ap.meta` with probeinterface) and lists the rats, the number of sessions, and per shank the number of channels and the
  depth range (y = distance from the tip, µm; small y = deep).
- **Depth coverage:** for every rat × shank × `BIN_UM` (100 µm) depth bin, counts how many final sessions record there
  (raw maps, before bad-channel removal). A bin with at least `MIN_SESS` (5) sessions is counted as testable. The figure shows
  the counts per rat (colour = number of sessions).

**Expected:** 12 electrode maps, all used by the 60 final sessions.

**Input:** `final_sessions.csv`. **Saves:** `electrode_maps.csv`, `depth_coverage.csv`, figure `depth_coverage.png`.

In [ ]:
maps = fs.electrode_maps(read("final_sessions"), RAW)
fxc.save_table(maps, "electrode_maps")

In [ ]:
cov, cov_summary, fig = fs.depth_coverage(read("final_sessions"), RAW, BIN_UM, MIN_SESS)
fxc.save_table(cov, "depth_coverage"); fxc.save_fig(fig, "depth_coverage"); plt.show()

## Step 9 · Task settings and tone delays per session

**What it does.**
- **Task settings:** for each final session, the distinct values of the setting columns of `trials.csv`
  (catch, vibration, long_prob, wt, real_wt, reward_size, tone frequency/amplitude/duration; more than 4 values are summarised
  as a range), whether `licks.npz` exists, and the % correct / early / late / long trials (all trials, before the trial filter).
- **Tone delays:** the measured tone − press delay of every trial with a tone, the number of trials with another delay than
  0.35 / 1.05 s, and the `long_prob` block structure (number of switches, trials per value).

**Expected (checked 2026-10-16):** catch and vibration always 0; `long_prob` changes in blocks within a session (median 7 switches);
delays exactly 0.35 / 1.05 s (not `real_wt` 0.32 / 1.02); tone 8 kHz, 0.2 s; `reward_size` units differ between task versions, so don't pool them.

**Input:** `final_sessions.csv`. **Saves:** `task_settings_per_session.csv`, `tone_delay_longprob_check.csv`.

In [ ]:
tasks = fs.task_settings(read("final_sessions"), PROC)
fxc.save_table(tasks, "task_settings_per_session")

In [ ]:
delays = fs.tone_delays(read("final_sessions"), PROC)
fxc.save_table(delays, "tone_delay_longprob_check")

## Step 10 · Probe location: brain region of every recorded channel

**What it does.** Each session has its own `channel_regions.csv` (and `channel_locations.xlsx`) in `share`, because the
electrode map changes between sessions. For every final session:
1. checks that the files exist and that the xlsx agrees with the csv;
2. reads the **recorded** channels of the 1000 Hz recording (after bad-channel removal) and their positions;
3. matches every channel to the file **by position**: file x = probe x + `X_OFFSET_UM` (27 µm), same depth. Channels without a
   match are labelled `NOT IN FILE`;
4. extra check: the channel **number** also points to the same depth in the file.

Result: one row per recorded channel with session, rat, channel id, x, y, region and brain coordinates (AP, ML, DV in mm).

**Expected (checked 2026-10-20):** all files present, 100 % matched by position in every session.

**Input:** `final_sessions.csv`. **Saves:** **`channel_regions_all.csv`** (read by 04–07) and `channel_regions_inventory.csv`.

In [ ]:
regions_inv, ch_reg = fs.channel_regions(read("final_sessions"), PROC, X_OFFSET_UM)
fxc.save_table(regions_inv, "channel_regions_inventory")
fxc.save_table(ch_reg, "channel_regions_all")

### Step 10b · Region consistency per rat

**What it does.** Within each rat, an electrode position (x, y) should belong to the same region in every session.
It counts the positions with more than one region across sessions (conflicts) and lists them, then shows the depth
range (y, µm from the tip) of every region per rat and shank.

**Expected:** 0 conflicts. **Input:** `channel_regions_all.csv`. **Saves:** nothing.

In [ ]:
consistency, y_range = fs.region_consistency(read("channel_regions_all"))

### Step 10c · Probe plots per rat

**What it does.** One figure per rat: one panel per session (the 4 shanks, every recorded channel as a square coloured by its
region; title = date and the number of PL and IL channels) and a last panel with all sessions together.

**Input:** `channel_regions_all.csv`. **Saves:** figures `probe_regions_rat_<rat>.png`.

In [ ]:
for rat, fig in fs.plot_probe_regions(read("channel_regions_all")).items():
    fxc.save_fig(fig, f"probe_regions_rat_{rat}", dpi=150); plt.show()

### Step 10d · Probe orientation and position in the brain

**What it does.**
- **Orientation:** for every session and shank (≥ 10 channels with coordinates): median ML and AP, top and bottom DV, and the
  tilt = angle of the change in ML along DV (slope of ML on DV). Then the ML and AP spread between the 4 shanks of a session.
  The mPFC layers run parallel to the midline, so a spread in ML means the shanks lie across layers, and a spread in AP means along the same layers.
- **Figure:** one column per rat; top = coronal view (ML × DV, dashed line = midline), bottom = sagittal view (AP × DV);
  every electrode position recorded in the rat's sessions, coloured by region.

**Input:** `channel_regions_all.csv`. **Saves:** `electrode_positions_brain.csv` and the figure `summary probe position in brain.png`, both in `PROBE_DIR`.

In [ ]:
positions, orientation, spread = fs.probe_orientation(read("channel_regions_all"))
fig = fs.plot_probe_brain(positions)
fxc.save_table(positions, "electrode_positions_brain", folder=PROBE_DIR)
fxc.save_fig(fig, "summary probe position in brain", folder=PROBE_DIR); plt.show()

## Step 11 · Histology

### Step 11a · Previews of the histology slides

**What it does.** For every `.czi` slide in `HIST_SRC`: reads it, takes the maximum over z (or the first plane of any other
extra dimension), down-samples it so the long side is at most 2500 px, and makes an RGB image: channel 0 (DAPI) in blue,
channel 1 (DiI, the probe track) in red. Each channel is scaled between its 1st and 99.5th percentile. A 1 mm scale bar is
added from the pixel size in the file.

**Saves:** one PNG per slide in `HIST_DIR`.

In [ ]:
from pathlib import Path
for path in sorted(Path(HIST_SRC).glob("*.czi")):
    rgb, px = fs.czi_preview(path)
    fig = fs.plot_czi(rgb, px, path.name)
    fxc.save_fig(fig, path.stem, folder=HIST_DIR, dpi=150); plt.show(); plt.close(fig)

### Step 11b · Atlas figure with the region colours

**What it does.** Takes an atlas-overlay image (`ATLAS_SRC`) and recolours PL, IL and Cg1 to the same colours as the probe
figures (PL blue, IL orange, Cg1 purple), keeping the shading. PL and IL are found by their original colours; Cg1 is a fixed
outline drawn on the original image. The image is then enlarged 4× for print.

**Saves:** `ATLAS_OUT` (PNG) in `HIST_DIR`.

In [ ]:
atlas = fs.recolor_atlas(ATLAS_SRC)
display(atlas.resize((atlas.width // 4, atlas.height // 4)))
if SAVE:
    os.makedirs(HIST_DIR, exist_ok=True); atlas.save(os.path.join(HIST_DIR, ATLAS_OUT + ".png"), dpi=(300, 300)); print("saved in", HIST_DIR)